In [1]:
import os
import sys
import json
import uuid
import re
import shutil
import glob
import time
import threading
from tqdm import tqdm
from typing import Union
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from workflow.utils.logger import logger

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
deepseek_key = "sk-a190351c49254931b74c0d54bde898f4"

project_dir = '/home/gary/CloudPointProcessing/20260214学校操场实验/'
flight_dir = '/mnt/d/CloudPointProcessing/PCGSPRO_1761030020/wappe2007@qq.com/'

# projects_json_file = os.path.join(project_dir, 'projects.json')

# if not os.path.isfile(projects_json_file) or not os.path.exists(projects_json_file):
#     with open(projects_json_file, 'w') as f:
#         json.dump({}, f)
#     logger.warning(f"projects.json不存在于{project_dir}, 创建一个新的projects.json")

In [3]:
# 异步线程池
threads = []
max_workers = 32
threads_pool = ThreadPoolExecutor(max_workers=max_workers)

In [4]:
# system_prompt = """
# # 角色
# 你是一个智能文件夹名称解析器，负责将高速公路监测点的文件夹名称转换为结构化JSON格式。

# # 输入输出规范
# ## 输入
# 1. 用户输入：待处理的项目文件夹名称

# ## 输出要求
# - 输出完整JSON结构
# - 不要输出如```json和```这样的包裹框，直接输出JSON
# - 完整项目名称由: 平台 (DJI，即大疆智图)、实验时间 (如202507261601，包含了年月日时分秒)、实验编号 (如205，该编号不一定连续，是多少就写多少)、“点云实验20250726”固定字样、扫描方式(重复扫描/非重复扫描)、飞行高度 (如80m)、飞行速度（如6ms）、激光脉冲角度 (如90度)、土方厚度 (如9cm或“原始”)依次组成
# - 不输出任何其他额外内容

# # 数据处理规则
# 0. 项目名称解释：
#    - 样例1："DJI_202507261601_205_点云实验20250726非重复扫描80m-6ms-90度-9cm"
#    - 平台：DJI
#    - 实验时间：202507261601
#    - 实验编号：205
#    - 固定字样：点云实验20250726
#    - 扫描方式：非重复扫描
#    - 飞行高度：80m
#    - 飞行速度：6ms
#    - 激光脉冲角度：90度
#    - 土方厚度：9cm


# 1. 名称解析：
#    - 示例1："DJI_202507261601_205_点云实验20250726非重复扫描80m-6ms-90度-9cm"
#      {
#        "name": "DJI_202507261601_205_点云实验20250726非重复扫描80m-6ms-90度-9cm",
#        "id": "205",
#        "time": "202507261601",
#        "scan_mode": "非重复扫描",
#        "height": "80m",
#        "speed": 6ms, 
#        "angle": "90度",
#        "thickness": "9cm"
#      }
#    - 示例2："DJI_202507261112_184_点云实验20250726重复扫描40m-6ms-90度-3cm"
#      {
#        "name": "DJI_202507261112_184_点云实验20250726重复扫描40m-6ms-90度-3cm",
#        "id": "184",
#        "time": "202507261112",
#        "scan_mode": "重复扫描",
#        "height": "40m",
#        "speed": 6ms, 
#        "angle": "90度",
#        "thickness": "3cm"
#      }
#    - 示例3："DJI_202507261012_177_点云实验20250726非重复扫描80m-3ms-90度-原始"
#      {
#        "name": "DJI_202507261012_177_点云实验20250726非重复扫描80m-3ms-90度-原始",
#        "id": "177",
#        "time": "202507261012",
#        "scan_mode": "非重复扫描",
#        "height": "80m",
#        "speed": 3ms, 
#        "angle": "90度",
#        "thickness": "原始"
#      }
#      - 示例4："DJI_202507261012_180_点云实验20250726非重复扫描40m-6ms-90度-原始"
#      {
#        "name": "DJI_202507261012_180_点云实验20250726非重复扫描40m-6ms-90度-原始",
#        "id": "180",
#        "time": "202507261012",
#        "scan_mode": "非重复扫描",
#        "height": "40m",
#        "speed": 6ms, 
#        "angle": "90度",
#        "thickness": "原始"
#      }

# 2. 字段处理：
#   - 请忽略固定字样，即"点云实验20250726"，不用作为单独的JSON字段，在"name"字段中存在即可
#   - "thickness"可能是具体的几厘米厚度，也可能是“原始”字样

# # 响应格式
# {
#   "name": "完整文件夹名称",
#   "id": "实验编号",
#   "time": "从完整项目中截取的包括年月日时分秒的时间",
#   "scan_mode": "扫描方式",
#   "height": "飞行高度",
#   "speed": "飞行速度", 
#   "angle": "激光脉冲角度",
#   "thickness": "土方厚度/原始"
# }
# """

In [5]:
from openai import OpenAI
client = OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com")
# client = OpenAI(api_key='AAAA', base_url="http://localhost:11434/v1")

def send_messages(messages):
    while True:
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=messages,
                timeout=60
            )
            return response.choices[0].message
        except Exception as e:
            logger.warning(f"请求API时出错，正在重试：{e}")
            time.sleep(1)

In [6]:
ln_lock = threading.Lock() # 用于创建符号链接的锁
def async_copy(
    src_path: str,
    dst_path: str,
    is_dir: bool = False,
    logger: logger = None,
    real_copy: bool = False
) -> None:
    """
    异步复制文件或创建符号链接的线程函数
    
    Args:
        src_path: 源文件/目录路径
        dst_path: 目标路径
        is_dir: 是否处理目录
        logger: 可选日志记录器
        real_copy: 
            True - 执行真实拷贝
            False - 创建符号链接(默认)
    """
    try:
        
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            
        if real_copy:
            # 真实拷贝模式
            if is_dir:
                shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            else:
                shutil.copy2(src_path, dst_path)
            log_msg = f"已完成拷贝: {src_path} -> {dst_path}"
        else:
            # 符号链接模式
            with ln_lock:
                if os.path.exists(dst_path):
                    if os.path.islink(dst_path):
                        os.unlink(dst_path)  # 删除现有链接
                    else:
                        raise FileExistsError(f"目标已存在且不是链接: {dst_path}")

                if is_dir:
                    os.symlink(src_path, dst_path, target_is_directory=True)
                else:
                    os.symlink(src_path, dst_path)
                log_msg = f"已创建符号链接: {src_path} -> {dst_path}"

        if logger:
            logger.debug(log_msg)

    except Exception as e:
        error_msg = f"文件操作失败: {e} [操作: {'拷贝' if real_copy else '链接'}, 路径: {src_path} -> {dst_path}]"
        if logger:
            logger.error(error_msg)
        else:
            print(error_msg)

def async_copy_file(src_path: str, dst_path: str, real_copy: bool = False):
    if not os.path.isfile(src_path):
        raise ValueError(f"源路径不是文件: {src_path}")

    threads.append(threads_pool.submit(
                            async_copy, 
                            src_path=src_path,
                            dst_path=dst_path,
                            is_dir=False,
                            logger=logger,
                            real_copy=real_copy
                        ))

def async_copy_dir(src_path: str, dst_path: str, real_copy: bool = False):
    if not os.path.isdir(src_path):
        raise ValueError(f"源路径不是目录: {src_path}")

    threads.append(threads_pool.submit(
                            async_copy, 
                            src_path=src_path,
                            dst_path=dst_path,
                            is_dir=True,
                            logger=logger,
                            real_copy=real_copy
                        ))


In [7]:
from workflow.checker.check_exif_data_with_exiftool import check_exif_data_with_exiftool
def generate_img_metadata_async(img_dir, logger):
    metadata_json_path = os.path.join(img_dir, 'metadata.json')
    if not os.path.isfile(metadata_json_path) or not os.path.exists(metadata_json_path):
        with open(metadata_json_path, 'w') as f:
            json.dump({}, f)
    
    with open(metadata_json_path, 'r') as f:
        metadata = json.load(f)
    
    # 创建线程锁，用于保护对metadata字典的并发访问
    metadata_lock = threading.Lock()
    metadata_threads = []
    def process_image(file, logger):
        if file.endswith(('.jpg', '.JPG')):
            try:
                exif_data = check_exif_data_with_exiftool(
                    os.path.join(img_dir, file),
                    is_print=False)
                
                img_metadata = {
                    'GPSLatitudeRef': str(exif_data['EXIF:GPSLatitudeRef']),
                    'GPSLongitudeRef': str(exif_data['EXIF:GPSLongitudeRef']),
                    'GPSLatitude': str(exif_data['EXIF:GPSLatitude']),
                    'GPSLongitude': str(exif_data['EXIF:GPSLongitude']),
                    'GPSMapDatum:': str(exif_data['EXIF:GPSMapDatum']),
                    'ImageWidth': str(exif_data['File:ImageWidth']),
                    'ImageHeight': str(exif_data['File:ImageHeight']),
                    'GimbalRollDegree': str(exif_data['XMP:GimbalRollDegree']),
                    'GimbalPitchDegree': str(exif_data['XMP:GimbalPitchDegree']),
                    'GimbalYawDegree': str(exif_data['XMP:GimbalYawDegree']),
                    'AbsoluteAltitude': str(exif_data['XMP:AbsoluteAltitude']),
                    'RelativeAltitude': str(exif_data['XMP:RelativeAltitude']),
                    'FocalLength': str(exif_data['EXIF:FocalLength']),
                    'Model': str(exif_data['EXIF:Model']),
                }
                
                # 使用锁保护对共享数据的访问
                with metadata_lock:
                    metadata[file] = img_metadata
            except Exception as e:
                logger.error(f"处理图片 {file} 时出错: {str(e)}")
    
    for file in os.listdir(img_dir):
        metadata_threads.append(threads_pool.submit(
                process_image, 
                file=file,
                logger=logger
            ))
    
    # 等待所有任务完成
    for thread in metadata_threads:
        thread.result()
    
    # 所有线程完成后，一次性写入文件
    with open(metadata_json_path, 'w') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=4)
    
    logger.info(f"成功生成 {img_dir} 的元数据文件")

In [8]:
def dji_windows_to_wsl_path(windows_path):
    """
    将DJI images_list.json中的Windows路径转换为WSL路径
    
    参数:
        windows_path (str): Windows格式的路径
        
    返回:
        str: Linux格式的路径
    """
    # 使用PureWindowsPath解析Windows路径（不检查实际文件是否存在）
    path = windows_path.split('/')
    drive = path[0].lower().split(':')[0]
    win_path = os.path.join('/mnt', drive, *path[1:])
    
    return win_path

In [9]:
enable_judgement = False

def check_exist_project(proj_name, proj_json, flight_date):
    response = None
    messages = [
                {
                    "role": "user",
                    "content": f"下面是你需要遵守的规则：{system_prompt}\n\n下面是已经存在的项目供你参考：{str(proj_json)}\n\n下面是你需要处理的项目名称：{str(proj_name)}"
                }
            ]

    while True:
        response = send_messages(messages)
        if response.content == "":
            logger.warning("API响应为空，重试")
            continue
    
        logger.info(f"API响应: {response.content}")

        
        if enable_judgement:
            logger.warning("接受or重试 [Y/n]")
            user_input = input().strip().lower()
            if user_input == 'y' or user_input == '':
                logger.info("已接受")
            elif user_input == 'n':
                logger.info("重试")
                continue

        content = response.content
        
        # 去除CoT模型中的<think>标签
        content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL)

        # 去除空白行（只包含空格、换行或制表符的行）
        lines = content.splitlines()
        non_empty_lines = [line for line in lines if line.strip()]
        content = "\n".join(non_empty_lines)

        try:
            decoder = json.JSONDecoder()
            responed_json, index = decoder.raw_decode(content)
            logger.info(f"成功从大模型解析项目 {proj_name} 的JSON")

            # 检查是否存在项目
            for key, value in proj_json.items():
                if all(value[pos] == responed_json[pos] for pos in ['road', 'direction', 'pos1', 'pos2', 'pos3', 'pos4', 'pos5']) and value['time'] == flight_date:
                    logger.warning(f"项目已存在，UUID：{key}")
                    return None, key
            return responed_json, None

        except Exception as e:
            logger.warning(f"解析JSON时出错: {e}, 正在重试")
            continue

        

In [10]:
def is_target_project(flight: str, flight_dir): # 在这里筛选需要导入的项目名

    # 定义日期起止范围，闭区间，可以手动调整是否具体到日
    start_date = datetime.strptime("20260214", "%Y%m%d")
    end_date = datetime.strptime("20260214", "%Y%m%d")

    # 定义要选中的关键词
    keywords = ["泼溅"]

    match_date = re.search(r"DJI_(\d{12})", flight)
    if match_date:
        flight_date = datetime.strptime(match_date.group(1)[:8], "%Y%m%d")
        # 判断日期是否在指定范围内，并且文件夹名中包含关键词
        # if start_date <= flight_date <= end_date and any(keyword.casefold() in flight.casefold() for keyword in keywords):
        # if any(keyword.casefold() in flight.casefold() for keyword in keywords):
        if start_date <= flight_date <= end_date:
            # logger.info(f"文件夹{flight}符合筛选要求")
            return True
        
    # 数量少的话可以直接指定目标文件名列表
    # target_flights = [
    #     "DJI_202601271413_010_清云湛江方向-K603-562-632-点云-实验0127重复",
    #     "DJI_202601271413_009_清云湛江方向-K603-562-632-点云-实验0127非重复",
    #     "DJI_202601271548_011_清云湛江方向-K603-562-632-点云-实验0127重复填土",
    #     "DJI_202601271548_012_清云湛江方向-K603-562-632-点云-实验0127非重复填土"
    # ]

    # if flight in target_flights:
    #     return True
    
    # if not any(keyword in flight for keyword in ['惠清']):
    #     logger.warning(f"非惠清: {flight}")
    #     return False

    # for keyword in ['仿地', '视觉', '三门匝道']:
    #     if keyword in flight:
    #         logger.warning(f"排除{keyword}项目: {flight}")
    #         return False
    
    if len(os.listdir(os.path.join(flight_dir, flight))) == 0:
        logger.warning(f"文件夹为空: {flight}")
        return False

    return False

In [11]:
def get_date_from_sig_filename(sig_filename):
    # 从SIG文件名中提取日期
    try:
        flight_date = datetime.strptime(os.path.basename(sig_filename).split('_')[1], "%Y%m%d%H%M%S")
        return flight_date
    except Exception as e:
        raise e

In [12]:
def import_point_cloud_project(flight_dir: Union[os.PathLike, str], flight: str, flight_date: datetime, raw_path: str, projects_json: dict=None, proj_uuid: str=None):
    """
    点云项目导入逻辑

    Return:
    bool: 导入是否成功
    """
    if projects_json is not None and proj_uuid is not None:
        proj_dir = '_'.join([projects_json[proj_uuid]['time'], projects_json[proj_uuid]['name']])
        with open(projects_json_file, 'w') as f:
            json.dump(projects_json, f, indent=4, ensure_ascii=False)
    else:
        proj_dir = flight
        
    if not os.path.exists(os.path.join(project_dir, proj_dir)):
        os.makedirs(os.path.join(project_dir, proj_dir))

    # 复制点云数据
    logger.info("开始复制las")
    try:
        src_path = os.path.join(flight_dir, flight, 'lidars', 'terra_las', "cloud_merged.las")
        assert os.path.exists(src_path) and os.path.isfile(src_path)
        dst_path = os.path.join(project_dir, proj_dir, 'raw', 'las',  'cloud_merged.las')
        if not os.path.exists(dst_path):
            async_copy_file(src_path, dst_path, real_copy=False)
        else:
            logger.warning(f"项目 {proj_dir} 于 {flight_date.strftime("%Y-%m-%d-%H-%M-%S")} 采集的las已存在，跳过")
    except Exception as e:
        logger.warning(f"cloud_merged.las文件不存在，可能还未解算或解算时没有选中点云合并")

    # 复制pnts、b3dms和dem文件夹  有问题
    for folder in ['pnts', 'b3dms', 'dem']:
        logger.info(f"开始复制{folder}")
        try:
            src_path = os.path.join(flight_dir, flight, 'lidars', f'terra_{folder}')
            assert os.path.exists(src_path) and os.path.isdir(src_path)
            dst_path = os.path.join(project_dir, proj_dir, 'raw', folder)
            if not os.path.exists(dst_path):
                async_copy_dir(src_path, dst_path, real_copy=False)
            else:
                logger.warning(f"项目 {proj_dir} 于 {flight_date.strftime("%Y-%m-%d-%H-%M-%S")} 采集的{folder}已存在，跳过")
        except Exception as e:
            logger.warning(f"terra_{folder}文件夹不存在，可能还未解算")
            return False

    # 复制原图
    logger.info(f"开始复制原图")
    img_dst_dir = os.path.join(project_dir, proj_dir, 'raw', 'img')
    
    if not os.path.exists(img_dst_dir):
        for file in os.listdir(raw_path):
            if file.endswith((".jpg", ".JPG")):
                src_path = os.path.join(raw_path, file)
                dst_path = os.path.join(project_dir, proj_dir, 'raw', 'img', file.split('.')[0] + ".jpg")
                if not os.path.exists(dst_path):
                    async_copy_file(src_path, dst_path, real_copy=False)

        threads.append(threads_pool.submit(
                    generate_img_metadata_async, 
                    img_dir=img_dst_dir,
                    logger=logger
                ))
    else:
        logger.warning(f"项目 {proj_dir} 于 {flight_date.strftime("%Y-%m-%d-%H-%M-%S")} 采集的原图已存在，跳过")

    return True


In [13]:
def import_image_project(flight_dir: Union[os.PathLike, str], flight: str, projects_json: dict, proj_uuid: str):
    """
    视觉项目导入逻辑

    Return:
    bool: 导入是否成功
    """
    if not os.path.exists(os.path.join(flight_dir, flight, 'images', 'sentry_database')):
        logger.warning(f"项目 {flight} 尚未解算，跳过")

    image_list_json_path = os.path.join(flight_dir, flight, 'images', 'survey', 'image_list.json')
    with open(image_list_json_path, 'r') as f:
        image_list_json = json.load(f)
    raw_path = os.path.dirname(dji_windows_to_wsl_path(image_list_json[0]['path']))
    sig_files = glob.glob(os.path.join(raw_path, '*.SIG'))
    if not sig_files:
        logger.warning(f"未找到SIG文件，无法确定点云巡飞时间，跳过: {flight}")
        return False
        
    try:
        flight_date = get_date_from_sig_filename(sig_files[0])
        logger.info(f"{flight} 解析到巡飞时间: {flight_date}")
    except Exception as e:
        logger.error(f"{flight} 解析巡飞时间失败: {e}")
        return False

    # 复制正射图片
    logger.info(f"开始复制正射图片")
    
    try:
        src_path = os.path.join(flight_dir, flight, 'map', f'result.tif')
        dst_path = os.path.join(project_dir, proj_uuid, 'raw', 'dom', flight_date.strftime("%Y-%m-%d-%H-%M-%S") + ".tif")
        if not os.path.exists(dst_path):
            async_copy_file(src_path, dst_path, real_copy=False)
        else:
            logger.warning(f"项目 {projects_json[proj_uuid]['name']} 于 {flight_date.strftime("%Y-%m-%d-%H-%M-%S")} 采集的正射图片已存在，跳过")
    except Exception as e:
        logger.warning(f"正射图片不存在，可能还未解算")

    # 复制原图
    logger.info(f"开始复制原图")
    img_dst_dir = os.path.join(project_dir, projects_json[proj_uuid]['name'], 'raw', 'img', flight_date.strftime("%Y-%m-%d-%H-%M-%S"))
    
    if not os.path.exists(img_dst_dir):
        for file in os.listdir(raw_path):
            if file.endswith((".jpg", ".JPG")):
                src_path = os.path.join(raw_path, file)
                dst_path = os.path.join(project_dir, projects_json[proj_uuid]['name'], 'raw', 'img', flight_date.strftime("%Y-%m-%d-%H-%M-%S"), file.split('.')[0] + ".jpg")
                if not os.path.exists(dst_path):
                    async_copy_file(src_path, dst_path, real_copy=False)

        threads.append(threads_pool.submit(
                    generate_img_metadata_async, 
                    img_dir=img_dst_dir,
                    logger=logger
                ))
    else:
        logger.warning(f"项目 {projects_json[proj_uuid]['name']} 于 {flight_date.strftime("%Y-%m-%d-%H-%M-%S")} 采集的原图已存在，跳过")

In [14]:
start_time = time.time()

for flight in tqdm(os.listdir(flight_dir)):

    if not os.path.isdir(os.path.join(flight_dir, flight)):
        continue

    if not is_target_project(flight, flight_dir):
        continue
    
    sentry_path = os.path.join(flight_dir, flight, 'sentry_database')
    processing_path = os.path.join(flight_dir, flight, 'processing')

    if not (os.path.exists(sentry_path) or os.path.exists(processing_path)):
    # if not os.path.exists(os.path.join(flight_dir, flight, 'sentry_database')):
        logger.warning(f"项目 {flight} 尚未解算，跳过")
        continue
    
    logger.info(f"开始处理项目: {flight}")
    
    # with open(projects_json_file, 'r') as f:
    #     projects_json = json.load(f)
        
    # 要先解析巡飞时间，后面check_exist_project函数中需要通过判断各字段是否相等来决定是否返回新的uuid
    # 对于同一个边坡，只有巡飞时间可以区分两次巡飞的数据
    lidar_list_json_path = os.path.join(flight_dir, flight, 'lidars', 'lidar_list.json')

    with open(lidar_list_json_path, 'r') as f:
        lidar_list_json = json.load(f)
    raw_path = dji_windows_to_wsl_path(lidar_list_json[0]['path'])
    sig_files = glob.glob(os.path.join(raw_path, '*.SIG'))
    
    if not sig_files:
        logger.warning(f"{flight} 未找到SIG文件，无法确定点云巡飞时间")

    try:
        flight_date = get_date_from_sig_filename(sig_files[0])
        logger.info(f"{flight} 解析到巡飞时间: {flight_date}")
    except Exception as e:
        logger.error(f"{flight} 解析巡飞时间失败: {e}")

    # 通过大模型自动解析和生成其他字段的具体值
    # json_value = check_exist_project(flight, projects_json, flight_date)

    # if json_value[0] is None:
    #     proj_uuid = json_value[1]
    # else:
    #     proj_uuid = str(uuid.uuid4())
    #     projects_json[proj_uuid] = json_value[0]
    #     projects_json[proj_uuid]['time'] = flight_date.strftime("%Y%m%d%H%M%S")  # 写入巡飞时间，方便后续使用

    # # 保存上述字段，包括巡飞时间等
    # with open(projects_json_file, 'w') as f:
    #     json.dump(projects_json, f, indent=4, ensure_ascii=False)

    if os.path.exists(os.path.join(flight_dir, flight, 'lidars')):
        logger.info(f"识别 {flight} 为点云巡飞")
        is_lidar = True
    elif os.path.exists(os.path.join(flight_dir, flight, 'images')):
        logger.info(f"识别 {flight} 为图片巡飞")
        is_lidar = False
    else:
        logger.warning(f"无法识别 {flight} 类型，跳过")
        continue

    if is_lidar:
        import_point_cloud_project(flight_dir, flight, flight_date, raw_path, 
                                #    projects_json, 
                                #    proj_uuid,
                                   )
    # else:
    #     import_image_project(flight_dir, flight, projects_json, proj_uuid)

logger.info("等待所有复制进程结束")
for thread in threads:
        thread.result()

logger.info("所有复制进程已结束")
end_time = time.time()
logger.info(f"总耗时: {end_time - start_time:.2f}秒")

  3%|▎         | 10/312 [00:00<00:06, 47.97it/s]

2026-02-24 15:35:50 - 3040859282 - WARNING - 文件夹为空: DJI_202508260914_495_石岭隧道清远端口点云
2026-02-24 15:35:50 - 3040859282 - WARNING - 文件夹为空: DJI_202508260943_500_石岭隧道惠州端口点云
2026-02-24 15:35:50 - 3040859282 - WARNING - 文件夹为空: DJI_202508270925_538_高山顶清远端口点云


 92%|█████████▏| 286/312 [00:04<00:00, 49.67it/s]

2026-02-24 15:35:54 - 2521782815 - INFO - 开始处理项目: DJI_202602141143_001_20260214学校操场水平-重复-无植被-无垫高
2026-02-24 15:35:54 - 2521782815 - INFO - DJI_202602141143_001_20260214学校操场水平-重复-无植被-无垫高 解析到巡飞时间: 2026-02-14 12:19:26
2026-02-24 15:35:54 - 2521782815 - INFO - 识别 DJI_202602141143_001_20260214学校操场水平-重复-无植被-无垫高 为点云巡飞
2026-02-24 15:35:54 - 2722286861 - INFO - 开始复制las
2026-02-24 15:35:54 - 2722286861 - INFO - 开始复制pnts
2026-02-24 15:35:54 - 2722286861 - INFO - 开始复制b3dms
2026-02-24 15:35:54 - 2722286861 - WARNING - terra_b3dms文件夹不存在，可能还未解算


 95%|█████████▌| 297/312 [00:04<00:00, 55.58it/s]

...

2026-02-24 15:35:55 - 2521782815 - INFO - 开始处理项目: DJI_202602141438_008_20260214学校操场倾斜-非重复-有植被
2026-02-24 15:35:55 - 2521782815 - INFO - DJI_202602141438_008_20260214学校操场倾斜-非重复-有植被 解析到巡飞时间: 2026-02-14 14:52:03
2026-02-24 15:35:55 - 2521782815 - INFO - 识别 DJI_202602141438_008_20260214学校操场倾斜-非重复-有植被 为点云巡飞
2026-02-24 15:35:55 - 2722286861 - INFO - 开始复制las
2026-02-24 15:35:55 - 2722286861 - INFO - 开始复制pnts
2026-02-24 15:35:55 - 2722286861 - INFO - 开始复制b3dms
2026-02-24 15:35:55 - 2722286861 - WARNING - terra_b3dms文件夹不存在，可能还未解算
2026-02-24 15:35:55 - 2521782815 - INFO - 开始处理项目: DJI_202602141438_009_20260214学校操场水平垫高-重复-有植被
2026-02-24 15:35:55 - 2521782815 - INFO - DJI_202602141438_009_20260214学校操场水平垫高-重复-有植被 解析到巡飞时间: 2026-02-14 15:06:34
2026-02-24 15:35:55 - 2521782815 - INFO - 识别 DJI_202602141438_009_20260214学校操场水平垫高-重复-有植被 为点云巡飞
2026-02-24 15:35:55 - 2722286861 - INFO - 开始复制las
2026-02-24 15:35:55 - 2722286861 - INFO - 开始复制pnts
2026-02-24 15:35:55 - 2722286861 - INFO - 开始复制b3dms
2026-02-24 15

100%|█████████▉| 311/312 [00:05<00:00, 56.47it/s]

2026-02-24 15:35:55 - 3040859282 - WARNING - 文件夹为空: 激光雷达点云项目


100%|██████████| 312/312 [00:05<00:00, 60.30it/s]

2026-02-24 15:35:55 - 2521782815 - INFO - 等待所有复制进程结束
2026-02-24 15:35:55 - 2521782815 - INFO - 所有复制进程已结束
2026-02-24 15:35:55 - 2521782815 - INFO - 总耗时: 5.20秒
